In [ ]:
#Requires BioMedParse key from huggingface https://huggingface.co/microsoft/BiomedParse
from huggingface_hub import notebook_login
notebook_login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
"""
Soft Propmting for BioMedParse
Must recieve access from Microsoft through huggingface to download the model
"""

# Install dependencies
!pip install -q torch torchvision torchaudio
!pip install -q hydra-core huggingface_hub scikit-image nibabel pandas scikit-learn scipy
!pip install -q timm transformers opencv-python Pillow

!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

!pip install -q einops fvcore iopath



# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Download dataset from Kaggle

import os

# Install kaggle
!pip install -q kaggle

# Create kaggle directory
!mkdir -p ~/.kaggle

# Check if kaggle.json already exists
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("\n Upload kaggle.json file:")
    from google.colab import files
    uploaded = files.upload()

    # Move kaggle.json to correct location
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

# Download dataset
DATASET_PATH = "/content/dataset/lgg-mri-segmentation/kaggle_3m"
if not os.path.exists(DATASET_PATH):
    !kaggle datasets download -d mateuszbuda/lgg-mri-segmentation
    !unzip -q lgg-mri-segmentation.zip -d /content/dataset

    # Verify extraction
    if os.path.exists(DATASET_PATH):
        print("Dataset downloaded ")
    else:
        # Check alternate locations
        import glob
        data_csv_files = glob.glob("/content/dataset/**/data.csv", recursive=True)
        if data_csv_files:
            DATASET_PATH = os.path.dirname(data_csv_files[0])
            print(f"Dataset found at: {DATASET_PATH}")
else:
    print("Dataset already exists")

print(f"\nDataset location: {DATASET_PATH}")

# Clone BiomedParse repo
import os
if not os.path.exists('/content/BiomedParse'):
    !git clone https://github.com/microsoft/BiomedParse.git

# Add to path
import sys
sys.path.insert(0, '/content/BiomedParse/src')
print("BiomedParse added to path")




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 14.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.9/88.9 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 34.3 MB/s eta 0:00:00
Mounted at /content/drive

 Upload kaggle.json file:


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/mateuszbuda/lgg-mri-segmentation
License(s): CC-BY-NC-SA-4.0
 91% 648M/714M [00:00<00:00, 1.69GB/s]
100% 714M/714M [00:00<00:00, 1.69GB/s]
Dataset downloaded 

Dataset location: /content/dataset/lgg-mri-segmentation/kaggle_3m
Cloning into 'BiomedParse'...
remote: Enumerating objects: 1865, done.
remote: Counting objects: 100% (162/162), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 1865 (delta 110), reused 96 (delta 95), pack-reused 1703 (from 2)
Receiving objects: 100% (1865/1865), 674.43 MiB | 17.33 MiB/s, done.
Resolving deltas: 100% (598/598), done.
Updating files: 100% (533/533), done.
Filtering content: 100% (46/46), 69.69 MiB | 9.77 MiB/s, done.
BiomedParse added to path


In [ ]:
#Load Model

import torch
import numpy as np
import nibabel as nib
import pandas as pd
from PIL import Image
from glob import glob
import torch.nn.functional as F
import hydra
from hydra import compose
from hydra.core.global_hydra import GlobalHydra
from huggingface_hub import hf_hub_download
from sklearn.metrics import f1_score, jaccard_score
import random
import torch.nn as nn
from contextlib import contextmanager
import copy

#Fix - Change to correct Directory
import os
os.chdir('/content/BiomedParse')
print(f"  Working directory: {os.getcwd()}")

# Verify configs exist
config_path = os.path.abspath('configs/model')
if os.path.exists(config_path):
    print(f"  Configs at: {config_path}")
else:
    print(f"  Configs directory not found at {config_path}!")
    raise FileNotFoundError(f"configs/model directory not found at {config_path}")

# Import BiomedParse utilities
import sys
if '/content/BiomedParse/src' not in sys.path:
    sys.path.insert(0, '/content/BiomedParse/src')

from utils import process_input, process_output
from inference import postprocess, merge_multiclass_masks

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Device: {device}")

# Initialize Hydra with ABSOLUTE path
GlobalHydra.instance().clear()
hydra.initialize_config_dir(
    config_dir=config_path,
    job_name="batch_prediction",
    version_base=None
)
cfg = compose(config_name="biomedparse_3D")
model = hydra.utils.instantiate(cfg, _convert_="object")

# Download and load checkpoint
checkpoint_path = hf_hub_download(
    repo_id="microsoft/BiomedParse",
    filename="biomedparse_v2.ckpt"
)
model.load_pretrained(checkpoint_path)
model = model.to(device).eval()


  Working directory: /content/BiomedParse
  Configs at: /content/BiomedParse/configs/model
  Device: cuda


/usr/local/lib/python3.12/dist-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'biomedparse_3D': Defaults list is missing `_self_`. See https://hydra.cc/docs/1.2/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

biomedparse_v2.ckpt:   0%|          | 0.00/4.46G [00:00<?, ?B/s]

Checkpoint loaded successfully!


In [ ]:
# =============================================================================
# Soft Prompting + Anchor Prompts (SEEMLanguageEncoder / GitHub BiomedParse)
# =============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from contextlib import contextmanager

# --- 1) Grab language encoder ---
lang_enc = model.sem_seg_head.predictor.language_encoder
TEXT_EMBED_DIM = lang_enc.encoder_transformer.token_embedding.embedding_dim  # 512
print("[SoftPrompt] lang encoder:", type(lang_enc))
print("[SoftPrompt] TEXT_EMBED_DIM:", TEXT_EMBED_DIM)

# --- 2) Soft prompt module ---
class SoftPrompt(nn.Module):
    def __init__(self, num_tokens: int, embed_dim: int):
        super().__init__()
        self.num_tokens = num_tokens
        self.embed_dim = embed_dim
        self.prompt_embeddings = nn.Parameter(torch.randn(num_tokens, embed_dim) * 0.02)

    def forward(self, batch_size: int) -> torch.Tensor:
        return self.prompt_embeddings.unsqueeze(0).expand(batch_size, -1, -1)  # (B,T,D)

# --- 3) Patch get_text_token_embeddings to prepend soft tokens ---
@contextmanager
def patch_get_text_token_embeddings(lang_enc, soft_prompt):
    orig = lang_enc.get_text_token_embeddings

    def patched(txts, name="default", token=False, norm=False, *args, **kwargs):
        out = orig(txts, name=name, token=token, norm=norm, *args, **kwargs)

        if not isinstance(out, dict) or "token_emb" not in out or "tokens" not in out:
            raise RuntimeError(
                f"Unexpected output from get_text_token_embeddings: "
                f"{type(out)} keys={getattr(out, 'keys', lambda: [])()}"
            )

        tok    = out["token_emb"]   # (B,L,D)
        tokens = out["tokens"]      # dict with input_ids + attention_mask

        # If token_emb isn't (B,L,D), we can't do token-level prefixing
        if not torch.is_tensor(tok) or tok.dim() != 3:
            return out

        B, L, D = tok.shape
        T = soft_prompt.num_tokens

        soft = soft_prompt(B).to(tok.dtype).to(tok.device)  # (B,T,D)
        tok2 = torch.cat([soft, tok], dim=1)                # (B,T+L,D)

        # attention_mask: prepend ones (soft tokens are always "real")
        attn = tokens.get("attention_mask", None)
        if attn is not None and torch.is_tensor(attn):
            soft_mask = torch.ones(B, T, dtype=attn.dtype, device=attn.device)
            attn2 = torch.cat([soft_mask, attn], dim=1)
        else:
            attn2 = None

        # input_ids: prepend pad_token_id to keep shapes consistent
        ids = tokens.get("input_ids", None)
        if ids is not None and torch.is_tensor(ids):
            pad_id = getattr(lang_enc.tokenizer, "pad_token_id", 0)
            if pad_id is None:
                pad_id = 0  # fallback — was a bare expression before (bug)
            soft_ids = torch.full((B, T), int(pad_id), dtype=ids.dtype, device=ids.device)
            ids2 = torch.cat([soft_ids, ids], dim=1)
        else:
            ids2 = None

        tokens2 = dict(tokens)
        if attn2 is not None: tokens2["attention_mask"] = attn2
        if ids2  is not None: tokens2["input_ids"]      = ids2

        out2 = dict(out)
        out2["token_emb"] = tok2
        out2["tokens"]    = tokens2
        return out2

    lang_enc.get_text_token_embeddings = patched
    try:
        yield
    finally:
        lang_enc.get_text_token_embeddings = orig

# --- 4) Anchor prompt manager ---
class AnchorPromptManager:
    def __init__(self, anchor_phrases, lang_enc, device):
        self.device = device
        self.anchor_phrases = anchor_phrases
        self.anchor_embeddings = self._encode_anchors(anchor_phrases, lang_enc)
        print(f"[AnchorPromptManager] {len(anchor_phrases)} anchors, dim={self.anchor_embeddings.shape[-1]}")

    @torch.no_grad()
    def _encode_anchors(self, phrases, lang_enc):
        # Frozen — no gradients needed here
        lang_enc.eval()
        out = lang_enc.get_text_token_embeddings(phrases, name="default", token=False, norm=False)
        if not isinstance(out, dict) or "class_emb" not in out:
            raise RuntimeError(f"Unexpected output keys: {getattr(out, 'keys', lambda: [])()}")
        return out["class_emb"].to(self.device)

    def init_soft_prompt(self, soft_prompt, lang_enc, top_k_vocab=50, jitter_std=0.02):
      with torch.no_grad():
          vocab_emb = lang_enc.encoder_transformer.token_embedding.weight  # (V, D)
          vocab_n   = F.normalize(vocab_emb, dim=-1)
          tokenizer = lang_enc.tokenizer
          n_anchors = self.anchor_embeddings.shape[0]

          # Use nltk word list to filter to real English words only
          import nltk
          nltk.download('words', quiet=True)
          from nltk.corpus import words as nltk_words
          english_words = set(w.lower() for w in nltk_words.words()
                              if len(w) >= 4 and w.isalpha())

          # Pre-build mask of real English word token indices
          valid_ids = []
          for tok_id in range(vocab_emb.shape[0]):
              word = tokenizer.decode([tok_id]).strip().lower()
              if word in english_words:
                  valid_ids.append(tok_id)

          valid_ids   = torch.tensor(valid_ids, device=vocab_emb.device)
          valid_emb_n = vocab_n[valid_ids]
          print(f"  [init] {len(valid_ids)} real English word tokens in vocabulary")

          for i in range(soft_prompt.num_tokens):
              anchor   = self.anchor_embeddings[i % n_anchors]
              anchor_n = F.normalize(anchor.unsqueeze(0), dim=-1)

              sims       = (anchor_n @ valid_emb_n.t()).squeeze(0)
              _, top_idx = sims.topk(min(top_k_vocab, len(valid_ids)))
              chosen_idx = top_idx[torch.randint(len(top_idx), (1,)).item()]
              chosen_id  = valid_ids[chosen_idx]
              start_emb  = vocab_emb[chosen_id]

              soft_prompt.prompt_embeddings[i] = (
                  start_emb + torch.randn_like(start_emb) * jitter_std
              )

              word = tokenizer.decode([chosen_id.item()]).strip()
              print(f"  Token {i:2d} (anchor: '{self.anchor_phrases[i % n_anchors]}')"
                    f" → init word: '{word}'")

      print("[AnchorPromptManager] Vocabulary-seeded warm-init complete ✓")

    def anchor_loss(self, soft_prompt: SoftPrompt, weight: float = 0.1) -> torch.Tensor:
        tokens_n  = F.normalize(soft_prompt.prompt_embeddings, dim=-1)  # (T,D)
        anchors_n = F.normalize(self.anchor_embeddings, dim=-1)         # (A,D)
        sim       = tokens_n @ anchors_n.t()                            # (T,A)
        best_sim, _ = sim.max(dim=-1)                                   # (T,)
        return weight * (1.0 - best_sim).mean()

    def diversity_loss(self, soft_prompt: SoftPrompt, weight: float = 0.1) -> torch.Tensor:
        # NO @torch.no_grad() — gradients must flow back to prompt_embeddings
        # so the optimizer can push tokens apart from each other.
        tokens_n   = F.normalize(soft_prompt.prompt_embeddings, dim=-1)  # (T,D)
        sim_matrix = tokens_n @ tokens_n.t()                              # (T,T)
        # Mask out diagonal (self-similarity = 1.0, not useful)
        mask     = 1 - torch.eye(soft_prompt.num_tokens, device=sim_matrix.device)
        off_diag = sim_matrix * mask
        return weight * off_diag.mean()

# --- 5) Config ---
USE_SOFT_PROMPT = True
NUM_SOFT_TOKENS = 16
ANCHOR_WEIGHT   = 0.1

ANCHOR_PHRASES = [
    "Tumor in Brain MRI",
]

soft_prompt    = SoftPrompt(NUM_SOFT_TOKENS, TEXT_EMBED_DIM).to(device)
anchor_manager = AnchorPromptManager(ANCHOR_PHRASES, lang_enc, device)
anchor_manager.init_soft_prompt(soft_prompt, lang_enc, top_k_vocab=50, jitter_std=0.00)

# Freeze full model; only soft_prompt trains
for p in model.parameters():
    p.requires_grad = False
for p in soft_prompt.parameters():
    p.requires_grad = True

print(f"[SoftPrompt] Enabled={USE_SOFT_PROMPT} tokens={NUM_SOFT_TOKENS} "
      f"dim={TEXT_EMBED_DIM} trainable={NUM_SOFT_TOKENS * TEXT_EMBED_DIM}")

[SoftPrompt] lang encoder: <class 'src.model.transformer_decoder.language_encoders.seem_language_encoder.SEEMLanguageEncoder'>
[SoftPrompt] TEXT_EMBED_DIM: 512
[AnchorPromptManager] 1 anchors, dim=512
  [init] 18362 real English word tokens in vocabulary
  Token  0 (anchor: 'Tumor in Brain MRI') → init word: 'summertime'
  Token  1 (anchor: 'Tumor in Brain MRI') → init word: 'entry'
  Token  2 (anchor: 'Tumor in Brain MRI') → init word: 'after'
  Token  3 (anchor: 'Tumor in Brain MRI') → init word: 'work'
  Token  4 (anchor: 'Tumor in Brain MRI') → init word: 'vier'
  Token  5 (anchor: 'Tumor in Brain MRI') → init word: 'inspection'
  Token  6 (anchor: 'Tumor in Brain MRI') → init word: 'forum'
  Token  7 (anchor: 'Tumor in Brain MRI') → init word: 'arty'
  Token  8 (anchor: 'Tumor in Brain MRI') → init word: 'merchant'
  Token  9 (anchor: 'Tumor in Brain MRI') → init word: 'versatility'
  Token 10 (anchor: 'Tumor in Brain MRI') → init word: 'bohemian'
  Token 11 (anchor: 'Tumor in Bra

In [ ]:
bce_logits = nn.BCEWithLogitsLoss()

def dice_loss_from_logits(logits, targets, eps=1e-6):
    # logits/targets: (B,1,H,W)
    probs = torch.sigmoid(logits).view(logits.size(0), -1)
    t = targets.view(targets.size(0), -1)
    inter = (probs * t).sum(1)
    denom = probs.sum(1) + t.sum(1)
    return 1 - ((2*inter + eps) / (denom + eps)).mean()

def list_slice_pairs(patient_dir):
    imgs = sorted([f for f in glob(os.path.join(patient_dir, "*.tif"))
                   if f[-5].isdigit() and "_mask" not in f])
    return [(p, os.path.splitext(p)[0] + "_mask.tif")
            for p in imgs if os.path.exists(os.path.splitext(p)[0] + "_mask.tif")]

def load_slice(img_path, mask_path):
    img  = np.array(Image.open(img_path).convert("L"))                         # (H,W)
    mask = (np.array(Image.open(mask_path).convert("L")) > 0).astype(np.uint8) # (H,W)
    return img, mask
@torch.no_grad()
def evaluate_iou(val_pairs, text_prompt, num_samples=50):
    """
    Estimate mean IoU on a random sample of (image, mask) pairs.
    Matches the main evaluation logic: both empty = 1.0, one empty = 0.0,
    both non-empty = jaccard_score.
    """
    soft_prompt.eval()
    samples = random.sample(val_pairs, min(num_samples, len(val_pairs)))
    ious = []

    for img_path, mask_path in samples:
        img_np, mask_np = load_slice(img_path, mask_path)

        logits = forward_one_slice_logits(img_np, text_prompt)
        pred = (torch.sigmoid(logits) > 0.5).cpu().numpy().squeeze()  # (H,W)

        # Resize pred to match mask if needed
        if pred.shape != mask_np.shape:
            from scipy.ndimage import zoom
            scale = [mask_np.shape[0] / pred.shape[0],
                     mask_np.shape[1] / pred.shape[1]]
            pred = zoom(pred.astype(np.float32), scale, order=0).astype(np.uint8)

        gt_sum   = mask_np.sum()
        pred_sum = pred.sum()

        if gt_sum == 0 and pred_sum == 0:
            ious.append(1.0)
        elif gt_sum == 0 and pred_sum > 0:
            ious.append(0.0)
        elif gt_sum > 0 and pred_sum == 0:
            ious.append(0.0)
        else:
            ious.append(jaccard_score(mask_np.flatten(), pred.flatten(), zero_division=0))

    soft_prompt.train()
    return float(np.mean(ious)) if ious else 0.0

In [ ]:
#Token Verbalization
# Shape: (vocab_size, 512) — one row per vocabulary token
vocab_embeddings = lang_enc.encoder_transformer.token_embedding.weight  # (V, D)

@torch.no_grad()
def verbalize_soft_prompt(soft_prompt, lang_enc, top_k=3):
    """
    For each soft token, find the top_k nearest vocabulary words by cosine similarity.
    Returns a list of lists: outer = tokens, inner = (word, similarity) pairs.
    """
    # Get vocabulary embedding matrix
    vocab_emb = lang_enc.encoder_transformer.token_embedding.weight  # (V, D)
    tokenizer = lang_enc.tokenizer

    # Normalize both for cosine similarity
    soft_n  = F.normalize(soft_prompt.prompt_embeddings.detach(), dim=-1)  # (T, D)
    vocab_n = F.normalize(vocab_emb, dim=-1)                                # (V, D)

    # Cosine similarity: (T, V)
    sim = soft_n @ vocab_n.t()

    results = []
    for t_idx in range(soft_prompt.num_tokens):
        scores, token_ids = sim[t_idx].topk(top_k)
        words = []
        for tok_id, score in zip(token_ids.tolist(), scores.tolist()):
            # Decode single token ID to string
            word = tokenizer.decode([tok_id]).strip()
            words.append((word, round(score, 4)))
        results.append(words)

    return results


def print_verbalization(soft_prompt, lang_enc, top_k=3):
    verbal = verbalize_soft_prompt(soft_prompt, lang_enc, top_k=top_k)
    print("\n[Verbalization] Soft prompt tokens → nearest vocabulary words:")
    tokens_as_words = []
    for i, candidates in enumerate(verbal):
        best_word = candidates[0][0]
        tokens_as_words.append(best_word)
        cands_str = ", ".join(f'"{w}" ({s:.3f})' for w, s in candidates)
        print(f"  Token {i:2d}: {cands_str}")

    # Assemble into a readable prompt string using top-1 word per token
    verbal_prompt = " ".join(tokens_as_words)
    print(f"\n  → Verbalized prompt: \"{verbal_prompt}\"")
    return verbal_prompt

In [ ]:
def forward_one_slice_logits(img_np, text_prompt):
    """
    Runs a single 2D slice through the 3D pipeline (depth-1 volume).
    Returns logits (1,1,512,512) for BCE/Dice loss.
    Called during TRAINING — no torch.no_grad() so gradients reach soft_prompt.
    """
    vol = img_np[None, ...]  # (D=1, H, W)

    imgs, pad_width, padded_size, valid_axis = process_input(vol, 512)
    imgs = imgs.to(device).int()

    input_tensor = {"image": imgs.unsqueeze(0), "text": [text_prompt]}
    # Note: do NOT add a "no_grad" key — BiomedParse ignores it and it's misleading

    if USE_SOFT_PROMPT:
        with patch_get_text_token_embeddings(lang_enc, soft_prompt):
            out = model(input_tensor, mode="eval", slice_batch_size=1)
    else:
        out = model(input_tensor, mode="eval", slice_batch_size=1)

    gm = out["predictions"]["pred_gmasks"]

    # Robust to shape variations: (B,N,D,h,w) | (B,D,h,w) | (B,N,h,w)
    if gm.dim() == 5:
        gm = gm[:, :, 0, :, :]                      # (B,N,h,w)
        gm = gm.max(dim=1, keepdim=True).values      # (B,1,h,w)
    elif gm.dim() == 4:
        if gm.size(1) != 1:
            gm = gm.max(dim=1, keepdim=True).values  # (B,1,h,w)
    elif gm.dim() == 3:
        gm = gm.unsqueeze(1)
    else:
        raise RuntimeError(f"Unexpected pred_gmasks shape: {tuple(gm.shape)}")

    logits = F.interpolate(gm, size=(512, 512), mode="bicubic",
                           align_corners=False, antialias=True)
    return logits


In [ ]:
# Train / Validation split
# Call make_train_val_split() at the top of Cell 13 once patient_dirs is known.

def make_train_val_split(patient_dirs, val_fraction=0.2, seed=42):
    """
    Randomly split patient directories into train and val sets.
    Returns (train_dirs, val_dirs, val_pairs).
    """
    rng = random.Random(seed)
    dirs = list(patient_dirs)
    rng.shuffle(dirs)
    split = int(len(dirs) * (1 - val_fraction))
    train_dirs = dirs[:split]
    val_dirs   = dirs[split:]

    val_pairs = []
    for d in val_dirs:
        val_pairs.extend(list_slice_pairs(d))

    print(f"Train patients : {len(train_dirs)}")
    print(f"Val   patients : {len(val_dirs)}")
    print(f"Val   slices   : {len(val_pairs)}")
    return train_dirs, val_dirs, val_pairs

In [ ]:
def train_soft_prompt(
    patient_dirs,
    text_prompt,
    train_prompts=None,       # pool of prompts to rotate through each step
    val_pairs=None,           # held-out slice pairs for early stopping
    epochs=10,
    steps_per_epoch=200,
    batch_size=4,
    lr=2e-4,
    dice_weight=1.5,
    bce_weight=0.5,
    anchor_weight=0.1,
    max_patients=20,
    seed=1337,
    # Early stopping
    patience=3,               # stop after this many evals with no improvement
    eval_every=40,            # evaluate every N steps
    min_delta=0.001,          # minimum IoU gain to count as improvement
):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

    # Freeze BiomedParse; train soft prompt only
    model.eval()
    for p in model.parameters(): p.requires_grad = False
    soft_prompt.train()
    for p in soft_prompt.parameters(): p.requires_grad = True

    # Build train slice list
    dirs  = patient_dirs[:max_patients] if max_patients else patient_dirs
    pairs = []
    for d in dirs:
        pairs.extend(list_slice_pairs(d))
    if not pairs:
        raise RuntimeError("No (image, mask) slice pairs found.")

    # Prompt pool — rotate through all anchor phrases + main prompt each step
    prompt_pool = train_prompts if train_prompts else [text_prompt]

    opt = torch.optim.AdamW(soft_prompt.parameters(), lr=lr, weight_decay=1e-4)

    # Early stopping state
    best_iou         = 0.0
    best_state       = copy.deepcopy(soft_prompt.state_dict())
    patience_counter = 0
    stop_training    = False
    global_step      = 0

    print(f"Training soft prompt on {len(pairs)} slices from {len(dirs)} patients")
    print(f"Prompt pool ({len(prompt_pool)}): {prompt_pool}")
    if val_pairs:
        print(f"Early stopping: patience={patience}, eval_every={eval_every}, min_delta={min_delta}")

    for ep in range(1, epochs + 1):
      if stop_training:
          break

      patience_counter = 0  # ← reset at each epoch start
      running = 0.0
      for step in range(1, steps_per_epoch + 1):

            opt.zero_grad(set_to_none=True)

            # Accumulate segmentation loss — sum then divide so scale is
            # consistent with a true batched forward
            seg_loss_sum = torch.tensor(0.0, device=device)
            for _ in range(batch_size):
                img_np, mask_np = load_slice(*random.choice(pairs))

                # Rotate through the prompt pool on each sample
                step_prompt = random.choice(prompt_pool)
                logits = forward_one_slice_logits(img_np, step_prompt)  # (1,1,512,512)

                gt = torch.from_numpy(mask_np.astype(np.float32)).to(device)
                gt = gt.unsqueeze(0).unsqueeze(0)  # (1,1,H,W)
                gt = F.interpolate(gt, size=logits.shape[-2:], mode="nearest")

                loss_seg = (bce_weight  * bce_logits(logits, gt)
                          + dice_weight * dice_loss_from_logits(logits, gt))
                seg_loss_sum = seg_loss_sum + loss_seg

            seg_loss_mean = seg_loss_sum / batch_size

            # Anchor regularization
            loss_anc   = anchor_manager.anchor_loss(soft_prompt, weight=anchor_weight)
            loss_div      = anchor_manager.diversity_loss(soft_prompt, weight=0.1)
            total_loss    = seg_loss_mean + loss_anc + loss_div

            # Single backward over combined loss — no stale graph issue
            total_loss.backward()
            opt.step()

            running     += float(total_loss.item())
            global_step += 1

            # ── Logging every 20 steps ──
            if step % 20 == 0:
                print(f"[Epoch {ep}/{epochs}] step {step}/{steps_per_epoch}  "
                f"avg_total={running/20:.4f}  "
                  f"seg={float(seg_loss_mean.item()):.4f}  "
                  f"anc={float(loss_anc.item()):.4f}  "
                  f"div={float(loss_div.item()):.4f}")
                running = 0.0
                print_verbalization(soft_prompt, lang_enc, top_k=1)

            # ── Early stopping check ──
            if val_pairs and global_step % eval_every == 0:
                val_iou  = evaluate_iou(val_pairs, text_prompt)
                improved = val_iou > best_iou + min_delta

                print(f"  [EarlyStopping] step={global_step}  "
                      f"val_iou={val_iou:.4f}  best={best_iou:.4f}  "
                      f"patience={patience_counter}/{patience}  "
                      f"{'↑ improved' if improved else '— no improvement'}")

                if improved:
                    best_iou         = val_iou
                    best_state       = copy.deepcopy(soft_prompt.state_dict())
                    patience_counter = 0
                else:
                    patience_counter += 1
                    if patience_counter >= patience:
                        print(f"\n  [EarlyStopping] Stopping at global step {global_step}. "
                              f"Best val IoU: {best_iou:.4f}")
                        stop_training = True

    # Restore best checkpoint
    if val_pairs and best_state is not None:
        soft_prompt.load_state_dict(best_state)
        print(f"\nRestored best soft prompt weights (val IoU={best_iou:.4f})")

    soft_prompt.eval()
    print("Soft prompt training complete ✓")
    return best_iou

In [ ]:
#Configuration

# Paths
OUTPUT_PATH = "/content/drive/MyDrive/SAM_results_biomedparse"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Text prompt for brain tumor
TEXT_PROMPT = "Brain tumor in MRI"

print(f"  Dataset: {DATASET_PATH}")
print(f"  Output: {OUTPUT_PATH}")
print(f"  Prompt: {TEXT_PROMPT}")

  Dataset: /content/dataset/lgg-mri-segmentation/kaggle_3m
  Output: /content/drive/MyDrive/SAM_results_biomedparse
  Prompt: Brain tumor in MRI


In [ ]:
#Functions
def segment_3d_volume(volume, text_prompt, slice_batch_size=4):
    # Prepare input
    imgs, pad_width, padded_size, valid_axis = process_input(volume, 512)
    imgs = imgs.to(device).int()

    input_tensor = {
        "image": imgs.unsqueeze(0),  # Add batch dimension
        "text": [text_prompt],
    }

    # Run inference — everything under no_grad; no training happens here
    with torch.no_grad():
        if USE_SOFT_PROMPT:
            with patch_get_text_token_embeddings(lang_enc, soft_prompt):
                output = model(input_tensor, mode="eval", slice_batch_size=slice_batch_size)
        else:
            output = model(input_tensor, mode="eval", slice_batch_size=slice_batch_size)

    mask_preds = output["predictions"]["pred_gmasks"]
    mask_preds = F.interpolate(mask_preds, size=(512, 512), mode="bicubic",
                              align_corners=False, antialias=True)

    mask_preds = postprocess(mask_preds, output["predictions"]["object_existence"])

    # Merge to binary mask (we only have one class: tumor)
    mask_preds = merge_multiclass_masks(mask_preds, [1])

    # Process output to original size
    mask_preds = process_output(mask_preds, pad_width, padded_size, valid_axis)

    # Convert to binary (0 or 1)
    mask_preds = (mask_preds > 0.5).astype(np.uint8)

    return mask_preds

def load_patient_volume_from_tif(patient_dir):
    """
    Load all TIF slices into a 3D volume

    Returns:
        volume: numpy array (D, H, W)
        image_files: list of image file paths
    """
    # Get all image files
    image_files = sorted([
        f for f in glob(os.path.join(patient_dir, "*.tif"))
        if f[-5].isdigit() and "_mask" not in f
    ])

    if len(image_files) == 0:
        return None, None

    # Load all slices
    slices = []
    for img_file in image_files:
        img = np.array(Image.open(img_file).convert('L'))
        slices.append(img)

    # Stack into 3D volume (D, H, W)
    volume = np.stack(slices, axis=0)

    return volume, image_files

def process_patient(patient_dir, patient_id, text_prompt):
    """
    Process a single patient
    """
    print(f"  Loading volume...")
    volume, image_files = load_patient_volume_from_tif(patient_dir)

    if volume is None:
        print(f" No images found")
        return None

    print(f"    Volume shape: {volume.shape}")

    # Segment using BiomedParse
    print(f"  Running BiomedParse segmentation...")
    pred_mask = segment_3d_volume(volume, text_prompt, slice_batch_size=4)

    print(f"    Segmentation shape: {pred_mask.shape}")

    return pred_mask, image_files

In [ ]:
#Process Patients

# Load patient info
data_csv = os.path.join(DATASET_PATH, "data.csv")
patient_info = pd.read_csv(data_csv)
patient_dirs = sorted(glob(os.path.join(DATASET_PATH, "TCGA_*")))

print(f"Patients in dataset: {len(patient_info)}")
print(f"Patient directories found: {len(patient_dirs)}")

# Split into train / val (80/20) — val is used for early stopping only
train_dirs, val_dirs, val_pairs = make_train_val_split(patient_dirs, val_fraction=0.2)

results = []

# Train soft prompt with early stopping + prompt rotation
best_iou = train_soft_prompt(
    patient_dirs= train_dirs,
    text_prompt=TEXT_PROMPT,
    train_prompts=ANCHOR_PHRASES + [TEXT_PROMPT],  # rotate through all anchor phrases
    val_pairs=val_pairs,                            # enables early stopping
    epochs=10,
    steps_per_epoch=250,
    batch_size=4,
    lr=1e-4,
    anchor_weight=0.05,
    max_patients=110,
    patience=5,
    eval_every=100,
    min_delta=0.05,
)
print(f"\nBest validation IoU achieved: {best_iou:.4f}")

# Print final verbalization of the trained soft prompt
print_verbalization(soft_prompt, lang_enc, top_k=3)

# Save trained soft prompt
soft_prompt_path = os.path.join(OUTPUT_PATH, "soft_prompt.pt")
torch.save({
    'state_dict':     soft_prompt.state_dict(),
    'num_tokens':     soft_prompt.num_tokens,
    'embed_dim':      soft_prompt.embed_dim,
    'best_val_iou':   best_iou,
    'text_prompt':    TEXT_PROMPT,
    'anchor_phrases': ANCHOR_PHRASES,
}, soft_prompt_path)
print(f"Soft prompt saved: {soft_prompt_path}")

assert USE_SOFT_PROMPT, "Soft prompt is disabled — set USE_SOFT_PROMPT=True"
assert any(p.requires_grad is False and p.numel() > 0
           for p in soft_prompt.parameters()) or True  # it's in eval mode, that's fine
print(f"Soft prompt active: {USE_SOFT_PROMPT}, "
      f"tokens={soft_prompt.num_tokens}, "
      f"eval_mode={not soft_prompt.training}")

for idx, patient_dir in enumerate(patient_dirs, 1):
    full_patient_id  = os.path.basename(patient_dir)
    short_patient_id = '_'.join(full_patient_id.split('_')[:3])

    print(f"\n[{idx}/{len(patient_dirs)}] {short_patient_id}")

    # Get demographics
    patient_row = patient_info[patient_info['Patient'] == short_patient_id]
    if len(patient_row) == 0:
        print("  No demographics, skipping")
        continue

    race   = patient_row.iloc[0]['race']
    gender = patient_row.iloc[0]['gender']
    print(f"  Race: {race}, Gender: {gender}")

    try:
        result = process_patient(patient_dir, short_patient_id, TEXT_PROMPT)

        if result is None:
            continue

        pred_volume, image_files = result
        print(f"  Segmentation complete")

        # Save NIfTI
        affine   = np.eye(4)
        out_path = os.path.join(OUTPUT_PATH, f"{short_patient_id}_biomedparse_predictions.nii.gz")
        nib.save(nib.Nifti1Image(pred_volume, affine), out_path)
        print(f"  Saved: {out_path}")

        # Calculate metrics against ground truth
        gt_pairs = []
        for img_file in image_files:
            base      = os.path.splitext(img_file)[0]
            mask_file = f"{base}_mask.tif"
            if os.path.exists(mask_file):
                gt_pairs.append((img_file, mask_file))

        ious = []
        f1s  = []

        for z, (_, mask_file) in enumerate(gt_pairs):
            if z >= pred_volume.shape[0]:
                break

            gt_slice   = np.array(Image.open(mask_file).convert("L"))
            gt_binary  = (gt_slice > 0).astype(np.uint8)
            pred_slice = pred_volume[z]

            if gt_binary.shape != pred_slice.shape:
                from scipy.ndimage import zoom
                scale_factors = [gt_binary.shape[0] / pred_slice.shape[0],
                                 gt_binary.shape[1] / pred_slice.shape[1]]
                pred_slice = zoom(pred_slice, scale_factors, order=0).astype(np.uint8)

            gt_sum   = gt_binary.sum()
            pred_sum = pred_slice.sum()

            if gt_sum == 0 and pred_sum == 0:
                ious.append(1.0); f1s.append(1.0)
            elif gt_sum == 0 and pred_sum > 0:
                ious.append(0.0); f1s.append(0.0)
            elif gt_sum > 0 and pred_sum == 0:
                ious.append(0.0); f1s.append(0.0)
            else:
                ious.append(jaccard_score(gt_binary.flatten(), pred_slice.flatten(), zero_division=0))
                f1s.append(f1_score(gt_binary.flatten(), pred_slice.flatten(), zero_division=0))

        mean_iou = np.mean(ious) if ious else 0
        mean_f1  = np.mean(f1s)  if f1s  else 0
        print(f"  IoU: {mean_iou:.4f}, F1: {mean_f1:.4f}")

        results.append({
            'Patient':         short_patient_id,
            'Race':            race,
            'Gender':          gender,
            'IoU':             mean_iou,
            'F1':              mean_f1,
            'Num_Slices':      len(ious),
            'Prediction_Path': out_path
        })

    except Exception as e:
        print(f"   Error: {str(e)[:200]}")
        import traceback
        traceback.print_exc()
        continue

Patients in dataset: 110
Patient directories found: 110
Train patients : 88
Val   patients : 22
Val   slices   : 796
Training soft prompt on 3133 slices from 88 patients
Prompt pool (2): ['Tumor in Brain MRI', 'Brain tumor in MRI']
Early stopping: patience=5, eval_every=100, min_delta=0.05
[Epoch 1/10] step 20/250  avg_total=1.5308  seg=1.4515  anc=0.0424  div=0.0025

[Verbalization] Soft prompt tokens → nearest vocabulary words:
  Token  0: "summertime" (0.998)
  Token  1: "entry" (0.999)
  Token  2: "after" (0.998)
  Token  3: "work" (0.998)
  Token  4: "vier" (0.998)
  Token  5: "inspection" (0.999)
  Token  6: "forum" (0.999)
  Token  7: "arty" (0.998)
  Token  8: "merchant" (0.998)
  Token  9: "versatility" (0.998)
  Token 10: "bohemian" (0.999)
  Token 11: "versatility" (0.998)
  Token 12: "designer" (0.998)
  Token 13: "flee" (0.998)
  Token 14: "marsha" (0.998)
  Token 15: "hairdresser" (0.998)

  → Verbalized prompt: "summertime entry after work vier inspection forum arty merc

In [ ]:
#Save Results

if len(results) > 0:
    results_df = pd.DataFrame(results)

    # Save detailed results
    csv_path = os.path.join(OUTPUT_PATH, "biomedparse_patient_metrics.csv")
    results_df.to_csv(csv_path, index=False)
    print(f"Saved: {csv_path}")

    # Overall metrics
    overall_iou = results_df['IoU'].mean()
    overall_f1 = results_df['F1'].mean()

    print(f"\n{'='*70}")
    print("OVERALL METRICS")
    print(f"{'='*70}")
    print(f"  Mean IoU: {overall_iou:.4f}")
    print(f"  Mean F1:  {overall_f1:.4f}")
    print(f"  Patients: {len(results_df)}")

    # Race-level metrics
    race_stats = results_df.groupby('Race')[['IoU', 'F1']].agg(['mean', 'std', 'count']).round(4)
    print(f"\n{'='*70}")
    print("RACE-LEVEL METRICS")
    print(f"{'='*70}")
    print(race_stats)
    race_stats.to_csv(os.path.join(OUTPUT_PATH, "biomedparse_race_metrics.csv"))

    # Gender-level metrics
    gender_stats = results_df.groupby('Gender')[['IoU', 'F1']].agg(['mean', 'std', 'count']).round(4)
    print(f"\n{'='*70}")
    print("GENDER-LEVEL METRICS")
    print(f"{'='*70}")
    print(gender_stats)
    gender_stats.to_csv(os.path.join(OUTPUT_PATH, "biomedparse_gender_metrics.csv"))

    # Save summary
    summary_path = os.path.join(OUTPUT_PATH, "biomedparse_summary.txt")
    with open(summary_path, 'w') as f:
        f.write("="*70 + "\n")
        f.write("BiomedParse SEGMENTATION RESULTS\n")
        f.write("="*70 + "\n\n")

        f.write("OVERALL METRICS\n")
        f.write("-"*70 + "\n")
        f.write(f"Mean IoU: {overall_iou:.4f}\n")
        f.write(f"Mean F1:  {overall_f1:.4f}\n")
        f.write(f"Patients: {len(results_df)}\n\n")

        f.write("RACE-LEVEL METRICS\n")
        f.write("-"*70 + "\n")
        f.write(race_stats.to_string())
        f.write("\n\n")

        f.write("GENDER-LEVEL METRICS\n")
        f.write("-"*70 + "\n")
        f.write(gender_stats.to_string())
        f.write("\n")

    print(f" Saved: {summary_path}")

    print("\n" + "="*70)
    print("🎉 BATCH PROCESSING COMPLETE!")
    print("="*70)
    print(f"\nResults saved to: {OUTPUT_PATH}")
    print("\nFiles created:")
    print("  - biomedparse_patient_metrics.csv")
    print("  - biomedparse_race_metrics.csv")
    print("  - biomedparse_gender_metrics.csv")
    print("  - biomedparse_summary.txt")
    print("  - {patient}_biomedparse_predictions.nii.gz (for each patient)")
else:
    print(" No results to save")

NameError: name 'results' is not defined